# Example 4: Agentic Data Analyst with MCP Context and Computation
Companion notebook for **Example 4** of Chapter 6.

A data analysis agent needs two things at once: business **context** and actual **computation**. MCP helps with both -- **resources** supply governed schema and metric definitions (so metric meaning comes from business definitions, not model guesswork), while **tools** run the actual queries and calculations. Flow: `load context -> profile -> aggregate/hypothesis -> summary`, with state carried across every node. Self-contained -- run this notebook on its own.

In [1]:
from typing import TypedDict


class AnalysisState(TypedDict, total=False):
    question: str
    dataset_ref: str
    schema: dict
    metric_defs: dict
    profile: dict
    aggregates: dict
    hypothesis_result: dict
    summary: str
    trail: list[dict]
    next_action: str

## A small MCP server: resources *and* tools
`schema://{dataset_ref}` and `metrics://{metric_name}` are **resources** (governed context to read, not actions to take). `profile_dataset` and `run_analysis` are **tools** (they compute something).

In [2]:
%%writefile analysis_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("AnalysisServer")

_SCHEMAS = {
    "sales.weekly_revenue": {
        "dataset": "sales.weekly_revenue",
        "columns": ["week_start", "region", "channel", "revenue", "orders", "returns", "sessions"],
    }
}
_METRIC_DEFS = {
    "revenue": {"metric": "revenue", "definition": "Sum of recognized revenue for a week after cancellations."}
}


@mcp.resource("schema://{dataset_ref}")
def schema_resource(dataset_ref: str) -> dict:
    """Expose dataset schema as an MCP resource."""
    return _SCHEMAS.get(dataset_ref, {"dataset": dataset_ref, "columns": []})


@mcp.resource("metrics://{metric_name}")
def metrics_resource(metric_name: str) -> dict:
    """Expose a governed metric definition as an MCP resource."""
    return _METRIC_DEFS.get(metric_name, {"metric": metric_name, "definition": "unknown"})


@mcp.tool()
def profile_dataset(dataset_ref: str) -> dict:
    """Profile a dataset: row count, missing rate, and dimension values."""
    return {
        "dataset_ref": dataset_ref,
        "row_count": 448,
        "missing_rate": 0.01,
        "regions": ["Northeast", "South", "West", "Midwest"],
        "channels": ["online", "retail", "partner"],
    }


@mcp.tool()
def run_analysis(question: str) -> dict:
    """Run a natural-language analytical query over the dataset."""
    q = question.lower()
    if "compare weekly revenue" in q:
        return {
            "window_weeks": 8,
            "region_change_pct": {"Northeast": -18.4, "South": 4.2, "West": 2.1, "Midwest": 1.3},
            "top_driver": {"region": "Northeast", "channel": "online", "change_pct": -24.6},
        }
    if "returns, discounting, or traffic" in q:
        return {
            "region": "Northeast",
            "channel": "online",
            "returns_change_pct": 3.1,
            "discount_rate_change_pct": 1.2,
            "sessions_change_pct": -19.7,
            "conclusion": "Traffic decline is the strongest explanation for the revenue drop.",
        }
    return {"note": "No matching analysis for this question."}


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing analysis_server.py


**Node 1 - `load_context`**: reads governed context via `read_resource`, not `call_tool` -- resources are for understanding, not taking action.

In [3]:
import sys, os

# Jupyter's stdout/stderr streams don't expose a real OS file descriptor, which crashes
# the MCP SDK's Windows subprocess-spawning fallback (it isn't needed outside Jupyter).
if os.name == "nt":
    try:
        sys.stderr.fileno()
    except Exception:
        sys.stderr = os.fdopen(os.dup(2), "w")

import asyncio
import json
import sys
from pathlib import Path
from typing import Any

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langgraph.graph import StateGraph, START, END

async def call_tool(session: ClientSession, name: str, arguments: dict) -> Any:
    """Call an MCP tool and unwrap its result into a plain Python value."""
    result = await session.call_tool(name, arguments)
    if result.structuredContent is not None:
        content = result.structuredContent
        return content.get("result", content) if isinstance(content, dict) else content
    text = result.content[0].text
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text


async def read_resource(session: ClientSession, uri: str) -> Any:
    """Read an MCP resource and unwrap it into a plain Python value."""
    result = await session.read_resource(uri)
    text = result.contents[0].text
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text

In [4]:
async def load_context(state: AnalysisState, mcp: ClientSession) -> AnalysisState:
    schema = await read_resource(mcp, f"schema://{state['dataset_ref']}")
    metric_defs = await read_resource(mcp, "metrics://revenue")
    return {"schema": schema, "metric_defs": metric_defs, "trail": [{"step": "load_context", "ok": True}]}

**Node 2 - `profile_data`**: now switches to `call_tool` -- profiling actually runs a computation over the dataset.

In [5]:
async def profile_data(state: AnalysisState, mcp: ClientSession) -> AnalysisState:
    profile = await call_tool(mcp, "profile_dataset", {"dataset_ref": state["dataset_ref"]})
    return {
        "profile": profile,
        "trail": state["trail"] + [{"step": "profile", "missing_rate": profile["missing_rate"]}],
    }

**Node 3 - `compute_aggregates`** and **Node 4 - `check_hypothesis`**: both call the same `run_analysis` tool with a different natural-language question.

In [6]:
async def compute_aggregates(state: AnalysisState, mcp: ClientSession) -> AnalysisState:
    result = await call_tool(mcp, "run_analysis", {
        "question": "Compare weekly revenue by region and channel for the last 8 weeks"
    })
    return {"aggregates": result, "trail": state["trail"] + [{"step": "aggregate", "result_ref": "aggregates"}]}


async def check_hypothesis(state: AnalysisState, mcp: ClientSession) -> AnalysisState:
    result = await call_tool(mcp, "run_analysis", {
        "question": "Check whether returns, discounting, or traffic explain the Northeast drop"
    })
    return {"hypothesis_result": result, "trail": state["trail"] + [{"step": "hypothesis", "result_ref": "hypothesis_result"}]}

**Routing**: branch to a hypothesis check only if the top driver looks like a real, sizeable drop.

In [7]:
def route_after_profile(state: AnalysisState) -> AnalysisState:
    missing_rate = state["profile"].get("missing_rate", 0.0)
    return {"next_action": "quality_check" if missing_rate > 0.05 else "aggregate"}


def route_after_aggregate(state: AnalysisState) -> AnalysisState:
    top_driver = state["aggregates"].get("top_driver", {})
    if top_driver.get("region") == "Northeast" and top_driver.get("change_pct", 0) < -10:
        return {"next_action": "hypothesis"}
    return {"next_action": "summary"}

**Final node**: produces both a short, decision-ready answer and the analytical trail -- the trail is what makes the result reviewable and trustworthy, not just a claim.

In [8]:
def summarize_analysis(state: AnalysisState) -> AnalysisState:
    aggregates = state.get("aggregates", {})
    hypothesis = state.get("hypothesis_result", {})
    top_driver = aggregates.get("top_driver", {})

    if hypothesis:
        summary = (
            f"Weekly revenue is down most in the Northeast, led by the {top_driver.get('channel')} channel. "
            f"Sessions changed by {hypothesis.get('sessions_change_pct')}% -- {hypothesis.get('conclusion')}"
        )
    else:
        summary = (
            f"Weekly revenue is down most in the Northeast, biggest drop in the "
            f"{top_driver.get('channel')} channel at {top_driver.get('change_pct')}%."
        )
    return {"summary": summary, "trail": state["trail"] + [{"step": "summary", "ok": True}]}

## Wiring the graph

In [9]:
def build_graph(mcp_session: ClientSession):
    graph = StateGraph(AnalysisState)

    async def load_context_node(state):
        return await load_context(state, mcp_session)

    async def profile_node(state):
        return await profile_data(state, mcp_session)

    async def aggregate_node(state):
        return await compute_aggregates(state, mcp_session)

    async def hypothesis_node(state):
        return await check_hypothesis(state, mcp_session)

    graph.add_node("load_context", load_context_node)
    graph.add_node("profile", profile_node)
    graph.add_node("route_after_profile", route_after_profile)
    graph.add_node("aggregate", aggregate_node)
    graph.add_node("route_after_aggregate", route_after_aggregate)
    graph.add_node("hypothesis", hypothesis_node)
    graph.add_node("summary", summarize_analysis)

    graph.add_edge(START, "load_context")
    graph.add_edge("load_context", "profile")
    graph.add_edge("profile", "route_after_profile")
    graph.add_conditional_edges(
        "route_after_profile", lambda s: s["next_action"],
        {"aggregate": "aggregate", "quality_check": "summary"},
    )
    graph.add_edge("aggregate", "route_after_aggregate")
    graph.add_conditional_edges(
        "route_after_aggregate", lambda s: s["next_action"],
        {"hypothesis": "hypothesis", "summary": "summary"},
    )
    graph.add_edge("hypothesis", "summary")
    graph.add_edge("summary", END)
    return graph.compile()

In [10]:
server_path = Path.cwd() / "analysis_server.py"
params = StdioServerParameters(command=sys.executable, args=[str(server_path)])

async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        app = build_graph(session)
        result = await app.ainvoke({
            "question": "Why did weekly revenue drop in the Northeast?",
            "dataset_ref": "sales.weekly_revenue",
        })

print("Summary:")
print(result["summary"])
print("\nAnalytical trail:")
for step in result["trail"]:
    print(f"  - {step}")

Summary:
Weekly revenue is down most in the Northeast, led by the online channel. Sessions changed by -19.7% -- Traffic decline is the strongest explanation for the revenue drop.

Analytical trail:
  - {'step': 'load_context', 'ok': True}
  - {'step': 'profile', 'missing_rate': 0.01}
  - {'step': 'aggregate', 'result_ref': 'aggregates'}
  - {'step': 'hypothesis', 'result_ref': 'hypothesis_result'}
  - {'step': 'summary', 'ok': True}
